# Strategy 1 — Genre Affinity

Strategy 1 walks one hop: from a user, through their 4★+ ratings, to the **genres** of those movies, then back out to other movies in those genres that the user hasn't rated. The score adds up *how strongly the user likes each matched genre*. The story for one rec is its `matched_genres` + a shared pool of `seed_movies` — Strategy 1's score is an aggregate over genres, not a single path, so a per-rec chain would mislead.

A few notes on what this notebook is and isn't:

- **Single-hop, not multi-hop.** User → seed → genre → rec. One person-shaped intermediary, not two.
- **Score = `sum(affinity)` across matched top genres.** A rec in 2 of the user's top genres outranks a rec in 1, weighted by how strong each affinity was.
- **Explainability shape = global seeds + per-rec `matched_genres`.** Up to 5 of the user's 4★+ titles in their top genres are surfaced as evidence the affinity is real — *shared* across all recs, not per-rec. Each rec also carries which of those top genres it sits in. Per-rec chains (`seed → person → rec`) land in Strategy 2.
- **Anchor user: 196.** Same recurring anchor as notebooks 09 / 10 / 12.
- **Notebook structure: build the query phase by phase first.** Cells 4–6 run Phase 1 / Phase 2 / Phase 3 as standalone Cypher with their own `RETURN`. Cells 7–9 then reveal the unified `GENRE_AFFINITY_CYPHER` constant and confirm the wrapper function returns the same rows.

In [1]:
# 1. Setup
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))

import pandas as pd
from graph import run_query, close_driver

ANCHOR_USER = 196
print(f"Anchor user: {ANCHOR_USER}")

Anchor user: 196


In [2]:
# 2. User 196's genre profile — what we expect Strategy 1 to surface
top_genres = run_query(
    """
    MATCH (u:User {userId: $userId})-[r:RATED]->(m:Movie)-[:IN_GENRE]->(g:Genre)
    WHERE r.rating >= 4.0
    RETURN g.name AS genre, count(m) AS affinity
    ORDER BY affinity DESC
    LIMIT 5
    """,
    {"userId": ANCHOR_USER},
)
print("Top 5 genres by 4★+ count:")
print(pd.DataFrame(top_genres))

Top 5 genres by 4★+ count:
        genre  affinity
0      Comedy        18
1       Drama         7
2     Romance         6
3         War         1
4  Children's         1


## Let's build the query phase by phase

Before showing the unified query (cell 8) and calling the wrapper function (cell 9), let's run each phase as its own Cypher block:

- **Phase 1 — top genres.** From the user's ≥4★ ratings, find their top 3 genres by count. Output: a small `(genre, affinity)` table.
- **Phase 2 — global seed titles.** For those top genres, surface up to 5 of the user's actual ≥4★ ratings as evidence the affinity is real. Output: one row per seed title, with the top genres it bridges. *Global*, not per-rec — Strategy 2 (notebook 12) is where seeds bridge to individual recs.
- **Phase 3 — forward walk.** For each top genre, find unseen recs in that genre, aggregate per rec, score by `sum(affinity)`, attach `matched_genres` and the global `seed_movies` list.

Each cell below ends with `RETURN` so it runs standalone. In the unified version (cell 8), those `RETURN`s become `WITH` — same callback to the `WITH` ↔ `RETURN` distinction from notebook 09's Cypher 101 walkthrough.

In [3]:
# 4. Phase 1 standalone — top 3 genres by 4★+ count
PHASE_1 = """
MATCH (u:User {userId: $userId})-[r:RATED]->(:Movie)-[:IN_GENRE]->(g:Genre)
WHERE r.rating >= $threshold
WITH u, g, count(*) AS affinity
ORDER BY affinity DESC
LIMIT $top_n_genres
RETURN g.name AS genre, affinity
ORDER BY affinity DESC
"""

phase_1_rows = run_query(
    PHASE_1,
    {"userId": ANCHOR_USER, "threshold": 4.0, "top_n_genres": 3},
)
print("Phase 1 — top genres:")
print(pd.DataFrame(phase_1_rows))

Phase 1 — top genres:
     genre  affinity
0   Comedy        18
1    Drama         7
2  Romance         6


In [4]:
# 5. Phase 2 standalone — global seed titles in those top genres
PHASE_2 = """
MATCH (u:User {userId: $userId})-[r:RATED]->(:Movie)-[:IN_GENRE]->(g:Genre)
WHERE r.rating >= $threshold
WITH u, g, count(*) AS affinity
ORDER BY affinity DESC
LIMIT $top_n_genres
WITH u, collect(g) AS top_genres

UNWIND top_genres AS top_g
MATCH (u)-[r2:RATED]->(seed:Movie)-[:IN_GENRE]->(top_g)
WHERE r2.rating >= $threshold
WITH seed, collect(DISTINCT top_g.name) AS in_genres
RETURN seed.title AS seed_title, in_genres
ORDER BY size(in_genres) DESC, seed_title
LIMIT 8
"""

phase_2_rows = run_query(
    PHASE_2,
    {"userId": ANCHOR_USER, "threshold": 4.0, "top_n_genres": 3},
)
print(f"Phase 2 — {len(phase_2_rows)} seed titles bridging user 196 to the top genres:")
print(pd.DataFrame(phase_2_rows))

Phase 2 — 8 seed titles bridging user 196 to the top genres:
                                          seed_title                 in_genres
0                            American President, The  [Comedy, Drama, Romance]
1  Adventures of Priscilla, Queen of the Desert, The           [Comedy, Drama]
2                                               Babe           [Comedy, Drama]
3                               English Patient, The          [Drama, Romance]
4               Mrs. Brown (Her Majesty, Mrs. Brown)          [Drama, Romance]
5                                   Muriel's Wedding         [Comedy, Romance]
6                               Nutty Professor, The         [Comedy, Romance]
7                                        Stand by Me           [Comedy, Drama]


In [5]:
# 6. Phase 3 standalone — walk forward to unseen recs, score, rank
PHASE_3 = """
MATCH (u:User {userId: $userId})-[r:RATED]->(:Movie)-[:IN_GENRE]->(g:Genre)
WHERE r.rating >= $threshold
WITH u, g, count(*) AS affinity
ORDER BY affinity DESC
LIMIT $top_n_genres
WITH u, collect({g: g, a: affinity}) AS scored_genres

UNWIND scored_genres AS sg
WITH u, scored_genres, sg.g AS top_g
MATCH (u)-[r2:RATED]->(seed:Movie)-[:IN_GENRE]->(top_g)
WHERE r2.rating >= $threshold
WITH u, scored_genres, collect(DISTINCT seed.title)[..5] AS seed_movies

UNWIND scored_genres AS sg
WITH u, seed_movies, sg.g AS top_g, sg.a AS top_a
MATCH (top_g)<-[:IN_GENRE]-(rec:Movie)
WHERE NOT EXISTS { (u)-[:RATED]->(rec) }
WITH rec, seed_movies,
     sum(top_a) AS score,
     collect(DISTINCT top_g.name) AS matched_genres
RETURN rec.movieId AS movieId,
       rec.title    AS title,
       rec.year     AS year,
       score,
       matched_genres,
       seed_movies
ORDER BY score DESC, title ASC
LIMIT $k
"""

phase_3_rows = run_query(
    PHASE_3,
    {"userId": ANCHOR_USER, "threshold": 4.0, "top_n_genres": 3, "k": 10},
)

def render(rows):
    if not rows:
        return pd.DataFrame(columns=["title", "year", "score", "matched_genres", "seed_movies"])
    df = pd.DataFrame(rows)
    df["matched_genres"] = df["matched_genres"].apply(lambda xs: ", ".join(xs))
    df["seed_movies"] = df["seed_movies"].apply(
        lambda xs: ", ".join(xs[:3]) + (" …" if len(xs) > 3 else "")
    )
    return df[["title", "year", "score", "matched_genres", "seed_movies"]]

render(phase_3_rows)

,title,year,score,matched_genres,seed_movies
0,Brassed Off,1996.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
1,Cinema Paradiso,1988.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
2,"Corrina, Corrina",1994.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
3,Don Juan DeMarco,1995.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
4,I Like It Like That,1994.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
5,Manhattan,1979.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
6,Something to Talk About,1995.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
7,Twelfth Night,1996.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
8,What Happened Was...,1994.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
9,Wings of Desire,1987.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"


## Now bring it all together

Phases 1–3 above are the body of the unified `GENRE_AFFINITY_CYPHER` constant in [`src/recommender.py`](../src/recommender.py). The collapse is mechanical:

- Each standalone phase ended with `RETURN` so it could run on its own.
- In the unified query, those `RETURN`s become `WITH` — the same variables flow into the next phase instead of leaving the query.
- The final `RETURN` lives at the end of Phase 3, producing the rec list.

Cell 8 prints the constant. Cell 9 calls the wrapper function and confirms the output matches Phase 3 above row-for-row.

In [6]:
# 8. The unified query — same Cypher you just ran phase by phase, chained with WITH
from recommender import GENRE_AFFINITY_CYPHER
print(GENRE_AFFINITY_CYPHER)


// Phase 1 — user's top genres by count of >= threshold ratings
MATCH (u:User {userId: $userId})-[r:RATED]->(:Movie)-[:IN_GENRE]->(g:Genre)
WHERE r.rating >= $threshold
WITH u, g, count(*) AS affinity
ORDER BY affinity DESC
LIMIT $top_n_genres
WITH u, collect({g: g, a: affinity}) AS scored_genres

// Phase 2 — global seeds: user's >= threshold titles in those top genres (up to 5)
UNWIND scored_genres AS sg
WITH u, scored_genres, sg.g AS top_g
MATCH (u)-[r2:RATED]->(seed:Movie)-[:IN_GENRE]->(top_g)
WHERE r2.rating >= $threshold
WITH u, scored_genres, collect(DISTINCT seed.title)[..5] AS seed_movies

// Phase 3 — unseen recs in top genres, scored + ranked
UNWIND scored_genres AS sg
WITH u, seed_movies, sg.g AS top_g, sg.a AS top_a
MATCH (top_g)<-[:IN_GENRE]-(rec:Movie)
WHERE NOT EXISTS { (u)-[:RATED]->(rec) }
WITH rec, seed_movies,
     sum(top_a) AS score,
     collect(DISTINCT top_g.name) AS matched_genres
RETURN rec.movieId AS movieId,
       rec.title AS title,
       rec.year AS ye

In [7]:
# 9. Call the wrapper — should match Phase 3 cell 6 row-for-row
from recommender import genre_affinity
recs = genre_affinity(ANCHOR_USER)
render(recs)

,title,year,score,matched_genres,seed_movies
0,Brassed Off,1996.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
1,Cinema Paradiso,1988.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
2,"Corrina, Corrina",1994.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
3,Don Juan DeMarco,1995.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
4,I Like It Like That,1994.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
5,Manhattan,1979.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
6,Something to Talk About,1995.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
7,Twelfth Night,1996.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
8,What Happened Was...,1994.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"
9,Wings of Desire,1987.0,31,"Comedy, Drama, Romance","Babe, Birdcage, The, Ace Ventura: Pet Detective …"


## Reading the explainability shape

Pick the top rec from cell 9. Read its `matched_genres` — that's the subset of user 196's top 3 genres this rec sits in. Read its `seed_movies` — those are up to 5 of user 196's own 4★+ ratings in those top genres, shared as evidence across every rec in the list.

> *"User 196's top genres are Comedy, Drama, Romance (cell 2). This rec is in all three (`matched_genres`). The score is the sum of user 196's affinity counts in those genres. `seed_movies` are user 196's own ≥4★ ratings in those genres — evidence the affinity isn't a one-rating fluke."*

Two things to internalize:

1. **`score` is a ranking signal, not a confidence percentage.** A score of 31 means *"sum of affinities across matched top genres = 31"* — useful for ordering, not for probability claims.
2. **`seed_movies` is global, not per-rec.** All recs in cell 9 share the same seed list. That's because Strategy 1's score aggregates over *genres*, not paths — there's no single `seed → genre → rec` chain that's load-bearing for any one rec, so attaching one would mislead. Strategy 2 (notebook 12) is where each rec gets its own per-rec chain, because *there* the structure of the query is a chain.

In [8]:
# 11. Knob — top_n_genres
narrow = genre_affinity(ANCHOR_USER, top_n_genres=1)
wide = genre_affinity(ANCHOR_USER, top_n_genres=5)

print("top_n_genres=1 — tightest per-rec story, all recs sit in the single dominant genre:")
print(render(narrow).head(3))
print("\ntop_n_genres=5 — wider candidate pool, bigger scores from multi-genre overlap:")
print(render(wide).head(3))

top_n_genres=1 — tightest per-rec story, all recs sit in the single dominant genre:
                     title    year  score matched_genres  \
0           101 Dalmatians  1996.0     18         Comedy   
1  8 Heads in a Duffel Bag  1997.0     18         Comedy   
2           A Chef in Love  1996.0     18         Comedy   

                                         seed_movies  
0  Babe, Birdcage, The, Ace Ventura: Pet Detective …  
1  Babe, Birdcage, The, Ace Ventura: Pet Detective …  
2  Babe, Birdcage, The, Ace Ventura: Pet Detective …  

top_n_genres=5 — wider candidate pool, bigger scores from multi-genre overlap:
              title    year  score          matched_genres  \
0       Brassed Off  1996.0     31  Comedy, Drama, Romance   
1   Cinema Paradiso  1988.0     31  Comedy, Drama, Romance   
2  Corrina, Corrina  1994.0     31  Comedy, Drama, Romance   

                                         seed_movies  
0  Babe, Birdcage, The, Ace Ventura: Pet Detective …  
1  Babe, Birdcag

In [9]:
# 12. Knob — threshold
permissive = genre_affinity(ANCHOR_USER, threshold=3.0)
strict = genre_affinity(ANCHOR_USER, threshold=4.5)

print("threshold=3.0 — more permissive seed pool, blurrier affinity signal:")
print(render(permissive).head(3))
print(f"\nthreshold=4.5 — sharper signal, smaller candidate pool. Returned {len(strict)} recs:")
print(render(strict).head(3) if strict else "(empty — user 196 doesn't have enough 4.5★+ ratings)")

threshold=3.0 — more permissive seed pool, blurrier affinity signal:
              title    year  score          matched_genres  \
0       Brassed Off  1996.0     47  Comedy, Drama, Romance   
1   Cinema Paradiso  1988.0     47  Comedy, Drama, Romance   
2  Corrina, Corrina  1994.0     47  Comedy, Drama, Romance   

                                      seed_movies  
0  Babe, Birdcage, The, While You Were Sleeping …  
1  Babe, Birdcage, The, While You Were Sleeping …  
2  Babe, Birdcage, The, While You Were Sleeping …  

threshold=4.5 — sharper signal, smaller candidate pool. Returned 10 recs:
              title    year  score          matched_genres  \
0       Brassed Off  1996.0     13  Comedy, Drama, Romance   
1   Cinema Paradiso  1988.0     13  Comedy, Drama, Romance   
2  Corrina, Corrina  1994.0     13  Comedy, Drama, Romance   

                                         seed_movies  
0  Babe, Ace Ventura: Pet Detective, Fish Called ...  
1  Babe, Ace Ventura: Pet Detective, Fis

In [10]:
# 13. Edge case — cold start
# 13a. A user with zero >=4* ratings, if one exists in the dataset
no_signal_user = run_query(
    """
    MATCH (u:User)
    WHERE NOT EXISTS { MATCH (u)-[r:RATED]->() WHERE r.rating >= 4 }
    RETURN u.userId AS userId LIMIT 1
    """,
)
if no_signal_user:
    cold_id = no_signal_user[0]["userId"]
    n = len(genre_affinity(cold_id))
    print(f"13a. User {cold_id} has zero >=4* ratings -> genre_affinity returned {n} recs.")
else:
    print("13a. (No >=4*-free user in MovieLens 100K — skipping.)")

# 13b. Non-existent user
n_ghost = len(genre_affinity(99999))
print(f"13b. User 99999 doesn't exist -> genre_affinity returned {n_ghost} recs.")

print("\nContract: empty list = no signal. Strategy 4 owns the fallback.")

13a. User 685 has zero >=4* ratings -> genre_affinity returned 0 recs.
13b. User 99999 doesn't exist -> genre_affinity returned 0 recs.

Contract: empty list = no signal. Strategy 4 owns the fallback.


In [11]:
# 14. Anchor sanity — none of the recs should already be rated by user 196
rec_ids = [r["movieId"] for r in recs]
overlap = run_query(
    """
    MATCH (u:User {userId: $userId})-[:RATED]->(m:Movie)
    WHERE m.movieId IN $ids
    RETURN count(m) AS overlap
    """,
    {"userId": ANCHOR_USER, "ids": rec_ids},
)
n = overlap[0]["overlap"] if overlap else -1
print("OK — 0 overlap" if n == 0 else f"MISMATCH — user 196 has already rated {n} of these recs")

OK — 0 overlap


In [12]:
# 15. Cleanup
close_driver()
print("Driver closed.")

Driver closed.
